# OverADP management-layer decomposition

## tl;dr

Across the 2023–2024 true-ADP simulations, the OverADP Target Intel draft created the largest relative advantage before any in-season moves: a 50.6% top-three rate versus 25.7% for ADP and +120.9 average points with the Week-1 lineup frozen. Weekly start/sit management added 44.9 points to OverADP teams, but it also helped opponents and narrowed the relative top-three lift. The current simple waiver rule added no incremental points beyond weekly lineup management, so a production waiver assistant needs a stronger role/injury/FAAB model.

## Context & Methods

The decision is whether OverADP's advantage originates in the draft and whether managers can recover additional value in season. Every strategy uses identical seasons, draft slots, random seeds, schedules, scoring, and management rules.

### Key Assumptions

- Primary evidence is 2023–2024 true Fantasy Football Calculator ADP; 2025 remains proxy sensitivity evidence.
- Frozen lineup means the preseason Week-1 lineup never changes and no waivers occur.
- Weekly lineups use preseason expectations plus only prior observed results.
- Full management adds one conservative same-position waiver move per team per week.
- Real weekly half-PPR results include injuries and absences through missed production.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
raw = pd.read_csv(RESULTS / 'management_decomposition_raw.csv')
summary = pd.read_csv(RESULTS / 'management_decomposition_summary.csv')
print(raw.shape)


## Data

Validate completeness, pairing, ranks, and primary-versus-proxy labeling before calculating lift.

In [ ]:
assert len(raw) == 13_500
assert not raw.duplicated(['season','episode','strategy','management_mode']).any()
assert raw.groupby(['season','strategy','management_mode']).size().eq(500).all()
assert raw['regular_rank'].between(1,12).all()
assert np.isfinite(raw[['regular_wins','points']]).all().all()
assert raw.groupby(['season','episode'])[['seed','draft_slot']].nunique().eq(1).all().all()
assert raw.loc[raw.season.isin([2023,2024]),'true_adp'].all()
assert not raw.loc[raw.season.eq(2025),'true_adp'].any()
print('Validation checks passed')


## Results

Pool the two true-ADP seasons with equal numbers of paired simulations.

In [ ]:
primary = raw[raw.true_adp].copy()
pooled = (primary.groupby(['management_mode','strategy']).agg(
    simulations=('episode','size'),
    avg_rank=('regular_rank','mean'),
    first_place_rate=('regular_rank',lambda s:(s==1).mean()),
    top3_rate=('regular_rank',lambda s:(s<=3).mean()),
    playoff_rate=('made_playoffs','mean'),
    championship_rate=('champion','mean'),
    points=('points','mean'),
).reset_index())
print(pooled.to_string(index=False))


In [ ]:
mode_order = ['frozen_lineup','weekly_lineups','lineups_plus_waivers']
wide = pooled.pivot(index='management_mode', columns='strategy')
lift = pd.DataFrame(index=mode_order)
lift['rank_improvement'] = wide['avg_rank']['adp'] - wide['avg_rank']['target_intel']
lift['top3_lift_pp'] = 100*(wide['top3_rate']['target_intel']-wide['top3_rate']['adp'])
lift['points_lift'] = wide['points']['target_intel']-wide['points']['adp']
lift['championship_lift_pp'] = 100*(wide['championship_rate']['target_intel']-wide['championship_rate']['adp'])
print(lift.round(2).to_string())


In [ ]:
def bootstrap_lift(frame, mode, reps=5000, seed=20260816):
    subset=frame[frame.management_mode.eq(mode)]
    rank_wide=subset.pivot(index=['season','episode'],columns='strategy',values='regular_rank')
    points_wide=subset.pivot(index=['season','episode'],columns='strategy',values='points')
    rank_delta=(rank_wide.adp-rank_wide.target_intel).to_numpy()
    point_delta=(points_wide.target_intel-points_wide.adp).to_numpy()
    rng=np.random.default_rng(seed); idx=rng.integers(0,len(rank_delta),size=(reps,len(rank_delta)))
    return {
      'mode':mode,
      'rank_lift':rank_delta.mean(),
      'rank_ci':np.quantile(rank_delta[idx].mean(axis=1),[.025,.975]),
      'points_lift':point_delta.mean(),
      'points_ci':np.quantile(point_delta[idx].mean(axis=1),[.025,.975]),
    }
bootstrap=[bootstrap_lift(primary,mode) for mode in mode_order]
for row in bootstrap: print(row)


## Visualization

The chart shows how the relative top-three advantage changes as every team receives more management help.

In [ ]:
labels=['Frozen lineup','Weekly lineups','Lineups + waivers']
x=np.arange(3); width=.34
adp=wide['top3_rate']['adp'].reindex(mode_order)
target=wide['top3_rate']['target_intel'].reindex(mode_order)
fig,ax=plt.subplots(figsize=(9,5.5))
ax.bar(x-width/2,adp,width,label='ADP',color='#7c8798')
ax.bar(x+width/2,target,width,label='Target Intel',color='#00d875')
ax.set_xticks(x,labels);ax.set_ylim(0,.60);ax.set_ylabel('Top-three regular-season rate')
ax.set_title('Target Intel remains ahead at every management level')
ax.legend(frameon=False);ax.grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(RESULTS/'management_top3_decomposition.png',dpi=180);plt.close(fig)
print(RESULTS/'management_top3_decomposition.png')


## Takeaways

1. The strongest evidence is a draft-created roster edge: Target Intel leads ADP even when the Week-1 lineup is frozen.
2. Active lineup management adds absolute points, but because every manager improves, it does not automatically increase OverADP's relative finish advantage.
3. The present waiver heuristic is too crude to claim incremental value. A useful production system should incorporate role changes, injuries, projected opportunity, schedule, positional replacement value, FAAB cost, and roster-specific need.
4. Championship outcomes remain weaker and noisier than regular-season top-three outcomes, so marketing should stay focused on better draft decisions rather than title guarantees.